# 🌍 Cross-Country Climate Comparison — COP32 Vulnerability Ranking

**Project:** EthioClimate Analytics — COP32 Position Paper  
**Branch:** `compare-countries`  
**Countries:** Ethiopia 🇪🇹 · Kenya 🇰🇪 · Sudan 🇸🇩 · Tanzania 🇹🇿 · Nigeria 🇳🇬  

---

## Objective
Synthesize cleaned data from all five countries to:
1. Compare temperature trends and precipitation variability
2. Count extreme heat days and consecutive dry days per year
3. Run statistical significance tests
4. **Produce a climate vulnerability ranking** to inform Ethiopia's COP32 position

## 0. Setup & Data Loading

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.visualize import (
    plot_multi_country_temperature,
    plot_precipitation_boxplots,
    plot_extreme_heat_days,
    set_style
)

pd.set_option('display.float_format', '{:.3f}'.format)

COUNTRIES = ['ethiopia', 'kenya', 'sudan', 'tanzania', 'nigeria']
COUNTRY_LABELS = {
    'ethiopia': 'Ethiopia',
    'kenya': 'Kenya',
    'sudan': 'Sudan',
    'tanzania': 'Tanzania',
    'nigeria': 'Nigeria'
}

print('✅ Imports successful')

In [ ]:
# Load all 5 cleaned CSVs and concatenate
dfs = []
for country in COUNTRIES:
    path = f'../data/{country}_clean.csv'
    df = pd.read_csv(path, parse_dates=['Date'])
    df['Country'] = COUNTRY_LABELS[country]
    dfs.append(df)
    print(f'  Loaded {country}: {len(df):,} rows')

df_all = pd.concat(dfs, ignore_index=True)
print(f'\nCombined dataset shape: {df_all.shape}')
df_all.head()

---
## 1. Temperature Trend Comparison

### 1.1 Multi-Country Monthly Average T2M (2015–2026)

In [ ]:
fig = plot_multi_country_temperature(df_all)
plt.show()

**📝 Interpretation:**  
_(Fill in — e.g., "Sudan consistently records the highest temperatures (>30°C annual mean), while Ethiopia shows the most variable seasonal pattern. All countries show an upward trend post-2020.")_

### 1.2 Temperature Summary Statistics

In [ ]:
temp_summary = df_all.groupby('Country')['T2M'].agg(
    Mean='mean',
    Median='median',
    Std='std',
    Min='min',
    Max='max'
).round(2).sort_values('Mean', ascending=False)

print('Temperature Summary (°C):')
display(temp_summary)

### 1.3 Warming Rate (Linear Trend per Country)

In [ ]:
from scipy.stats import linregress

warming_rates = []
for country, group in df_all.groupby('Country'):
    annual = group.groupby('YEAR')['T2M'].mean().reset_index()
    slope, intercept, r, p, se = linregress(annual['YEAR'], annual['T2M'])
    warming_rates.append({
        'Country': country,
        'Trend (°C/year)': round(slope, 4),
        'Trend (°C/decade)': round(slope * 10, 3),
        'R²': round(r**2, 3),
        'p-value': round(p, 4),
        'Significant (p<0.05)': '✅' if p < 0.05 else '❌'
    })

warming_df = pd.DataFrame(warming_rates).sort_values('Trend (°C/decade)', ascending=False)
print('Linear Warming Trends (2015–2026):')
display(warming_df)

**📝 Warming Rate Interpretation:**  
_(Fill in — e.g., "Sudan is warming at X°C/decade, the fastest among the five countries, while Nigeria shows the slowest trend.")_

---
## 2. Precipitation Variability Comparison

### 2.1 Side-by-Side Boxplots

In [ ]:
fig = plot_precipitation_boxplots(df_all)
plt.show()

### 2.2 Precipitation Summary Statistics

In [ ]:
precip_summary = df_all.groupby('Country')['PRECTOTCORR'].agg(
    Mean='mean',
    Median='median',
    Std='std',
    Max='max',
    Wet_Days=lambda x: (x > 1).sum()
).round(2).sort_values('Std', ascending=False)

print('Precipitation Summary (mm/day):')
display(precip_summary)

**📝 Interpretation:**  
_(Fill in — e.g., "Tanzania has the highest precipitation variability (Std = X mm/day), indicating highly erratic rainfall. Sudan has the lowest median precipitation, confirming its arid classification.")_

---
## 3. Extreme Event Frequency

### 3.1 Extreme Heat Days (T2M_MAX > 35°C) per Year

In [ ]:
heat_counts = df_all[df_all['T2M_MAX'] > 35].groupby(
    ['Country', 'YEAR']
).size().reset_index(name='extreme_heat_days')

# Fill missing years with 0
all_years = df_all['YEAR'].unique()
all_countries = df_all['Country'].unique()
idx = pd.MultiIndex.from_product([all_countries, all_years], names=['Country', 'YEAR'])
heat_counts = heat_counts.set_index(['Country', 'YEAR']).reindex(idx, fill_value=0).reset_index()

fig = plot_extreme_heat_days(heat_counts)
plt.show()

In [ ]:
# Summary table: total extreme heat days per country
heat_summary = heat_counts.groupby('Country')['extreme_heat_days'].agg(
    Total='sum', Mean_Per_Year='mean', Max_In_Year='max'
).sort_values('Total', ascending=False)
display(heat_summary)

### 3.2 Consecutive Dry Days per Year (PRECTOTCORR < 1mm)

In [ ]:
def max_consecutive_dry(series):
    """Return the maximum consecutive dry-day streak in a series."""
    is_dry = (series < 1).astype(int)
    max_streak = 0
    current = 0
    for val in is_dry:
        if val:
            current += 1
            max_streak = max(max_streak, current)
        else:
            current = 0
    return max_streak

dry_spells = df_all.groupby(['Country', 'YEAR']).apply(
    lambda g: max_consecutive_dry(g['PRECTOTCORR']), include_groups=False
).reset_index(name='max_dry_streak_days')

# Plot
set_style()
from src.visualize import COUNTRY_COLORS
fig, ax = plt.subplots(figsize=(13, 5))
width = 0.15
countries = dry_spells['Country'].unique()
years = sorted(dry_spells['YEAR'].unique())
x = np.arange(len(years))

for i, country in enumerate(countries):
    subset = dry_spells[dry_spells['Country'] == country].set_index('YEAR').reindex(years, fill_value=0)
    ax.bar(x + i * width, subset['max_dry_streak_days'],
           width=width, label=country, color=COUNTRY_COLORS.get(country, '#fff'), alpha=0.85)

ax.set_xticks(x + width * 2)
ax.set_xticklabels(years)
ax.set_title('Maximum Consecutive Dry Days per Year (PRECTOTCORR < 1 mm)', fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Max Consecutive Dry Days')
ax.legend(facecolor='#1e293b', edgecolor='#334155')
ax.grid(True, axis='y')
plt.tight_layout()
plt.show()

---
## 4. Statistical Testing — Is the Difference Real?

We run a **Kruskal–Wallis test** (non-parametric ANOVA) on daily T2M values across the 5 countries to assess whether their temperature distributions are statistically significantly different.

> **Why Kruskal–Wallis?** Temperature data is not perfectly normally distributed. Kruskal–Wallis makes no assumptions about the distribution shape.

In [ ]:
groups = [group['T2M'].dropna().values
          for _, group in df_all.groupby('Country')]

stat, p_value = stats.kruskal(*groups)

print(f'Kruskal–Wallis H-statistic: {stat:.2f}')
print(f'p-value: {p_value:.2e}')
print()
if p_value < 0.05:
    print('✅ SIGNIFICANT: Temperature distributions differ significantly across countries (p < 0.05).')
    print('   This means country-level differences in warming are statistically real — not random noise.')
else:
    print('❌ NOT significant at p < 0.05.')

---
## 5. Climate Vulnerability Ranking

We rank each country across 4 dimensions:
- **Warming rate** (°C/decade) — higher = more vulnerable
- **Temperature mean** — higher = more heat stress
- **Precipitation variability** (Std of PRECTOTCORR) — higher = more erratic rainfall
- **Extreme heat days** (total count) — higher = more exposure

In [ ]:
# Build composite vulnerability score
scores = {}

# Warming rate rank (from warming_df)
warming_rank = warming_df[['Country', 'Trend (°C/decade)']].set_index('Country')
warming_rank['warming_rank'] = warming_rank['Trend (°C/decade)'].rank(ascending=False)

# Temp mean rank
temp_rank = temp_summary[['Mean']].rename(columns={'Mean': 'temp_mean'})
temp_rank['temp_rank'] = temp_rank['temp_mean'].rank(ascending=False)

# Precip variability rank
precip_rank = precip_summary[['Std']].rename(columns={'Std': 'precip_std'})
precip_rank['precip_rank'] = precip_rank['precip_std'].rank(ascending=False)

# Heat days rank
heat_rank = heat_summary[['Total']].rename(columns={'Total': 'heat_days_total'})
heat_rank['heat_rank'] = heat_rank['heat_days_total'].rank(ascending=False)

# Combine
vuln = warming_rank.join(temp_rank).join(precip_rank).join(heat_rank)
vuln['composite_score'] = vuln[['warming_rank', 'temp_rank', 'precip_rank', 'heat_rank']].mean(axis=1)
vuln['vulnerability_rank'] = vuln['composite_score'].rank().astype(int)
vuln_sorted = vuln.sort_values('vulnerability_rank')

print('🏆 Climate Vulnerability Ranking (1 = Most Vulnerable):')
display(vuln_sorted[['Trend (°C/decade)', 'temp_mean', 'precip_std', 'heat_days_total', 'vulnerability_rank']])

---
## 6. COP32 Position Paper Bullets

### 🗣️ Five Negotiation-Grade Findings

_(Fill in after running all cells above with your actual data values)_

1. **Fastest Warming Country:**  
   *"[Country X] is warming at [X]°C/decade — the fastest rate among the five nations studied. At this trajectory, temperatures will exceed safe agricultural thresholds within [N] years, threatening food security for [X million] people."*

2. **Most Unstable Precipitation:**  
   *"[Country X] exhibits the highest precipitation variability (Std = [X] mm/day), making seasonal planning impossible for smallholder farmers. Year-to-year rainfall swings of up to [X]% demand immediate investment in drip irrigation and rainwater harvesting."*

3. **Extreme Heat & Drought Stress:**  
   *"[Country X] recorded [X] extreme heat days (T2M_MAX > 35°C) over 2015–2026, while [Country Y] experienced maximum dry streaks of [X] consecutive days in [year] — the longest in the dataset. Together, these metrics define a climate stress profile that demands loss-and-damage compensation under Article 8 of the Paris Agreement."*

4. **Ethiopia's Climate Profile:**  
   *"Ethiopia ranks [X] out of 5 in composite vulnerability. While not the hottest nation in the dataset, its combination of [bimodal rainfall / rapid warming / highland agriculture dependency] makes it disproportionately affected by even small temperature deviations. As COP32 host, Ethiopia must demonstrate that its own vulnerability is data-verified, not rhetorical."*

5. **Priority Finance Candidate:**  
   *"The data positions [Country X] as the most compelling case for priority climate finance. With [X]°C/decade warming, [X] extreme heat days, and maximum dry streaks of [X] days, [Country X] meets all three criteria — trend + impact + urgency — that qualify a country for frontline adaptation funding under the Bridgetown Initiative framework."*